# DPO Data Generation — v2

Generates a new 3k DPO training dataset with richer negative types.

**Positive side:** reuses `sft_data_ef.jsonl` — teacher-generated justifications that lead with a direct quote from the passage before connecting to the verdict. Stronger grounding than v1 positives.

**Negative taxonomy (6 types, 3000 total):**

| Type | What it targets | Source | Count |
|------|----------------|--------|-------|
| NEG1 — Wrong label | Label correctness | Reuse `dpo_neg1_3k.jsonl` | 750 |
| NEG2 — Bad reasoning | Overreach / fabrication | Reuse `dpo_neg2_3k.jsonl` | 750 |
| NEG3 — Label hedging | Decisiveness | Constructed | 250 |
| NEG4 — Circular citing | Passage grounding | GPT-4.1-mini | 500 |
| NEG5 — Degenerate output | Repetition / garbage | Constructed | 250 |
| NEG6 — Reasoning-label mismatch | Internal consistency | Constructed | 500 |
| **Total** | | | **3000** |

NEG6 is the most important new type: the model reasons correctly to the right verdict but then outputs the wrong label — exactly the failure seen in the dpo_c3 Ernest Medina example. DPO is well-suited to penalize this because the contrast is clear: same justification, right label (chosen) vs wrong label (rejected).

In [ ]:
import os
import json
import time
import random
from collections import Counter, defaultdict
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

RESOURCE_GROUP = "cis-5270-team-10"
OPENAI_API_KEY = ""

OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = ""

os.environ["AZURE_SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["AZURE_RESOURCE_GROUP"]  = "CIS-5270"
os.environ["AZURE_AOAI_ACCOUNT"]    = RESOURCE_GROUP
os.environ["AZURE_OPENAI_API_KEY"]  = OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = OPENAI_ENDPOINT

CREDENTIAL = DefaultAzureCredential()

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)

TEACHER_DEPLOYMENT = "gpt-4.1-mini"

random.seed(42)

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def save_jsonl(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Connected to Azure OpenAI")

Connected to Azure OpenAI


---
## Load Source Data

In [2]:
# positive side: EF-style justifications (direct passage quote + connection to verdict)
ef_data = load_jsonl("data/generated/sft_data_ef.jsonl")
print(f"EF data (chosen pool): {len(ef_data)}")
print(Counter(ex["label"] for ex in ef_data))

# neg1: wrong label + stolen justification (no API calls)
neg1_raw = load_jsonl("data/generated/dpo_neg1_3k.jsonl")
print(f"\nNEG1 pool: {len(neg1_raw)}")

# neg2: correct label + hallucinated reasoning (already generated)
neg2_raw = load_jsonl("data/generated/dpo_neg2_3k.jsonl")
print(f"NEG2 pool: {len(neg2_raw)}")

# index ef_data by id for chosen lookup
ef_by_id = {ex["id"]: ex for ex in ef_data}

EF data (chosen pool): 3000
Counter({'NOT MENTIONED': 1000, 'SUPPORTED': 1000, 'CONTRADICTED': 1000})

NEG1 pool: 1500
NEG2 pool: 1500


---
## NEG3 — Label Hedging (Constructed)

Take a correct response and wrap the label in uncertain language:
- "It could be SUPPORTED" / "possibly CONTRADICTED" / "might be NOT MENTIONED"
- Chosen is the same justification with a confident, correctly-formatted label.

Targets the model's tendency to hedge when it should commit.

In [4]:
HEDGE_TEMPLATES = [
    "It could be {label}: {justification}",
    "Possibly {label}: {justification}",
    "This might be {label}: {justification}",
    "I think {label}: {justification}",
    "Likely {label}: {justification}",
    "This seems to be {label}: {justification}",
]

neg3_pool = random.sample(ef_data, 250)
neg3_records = []
for ex in neg3_pool:
    template = random.choice(HEDGE_TEMPLATES)
    rejected = template.format(label=ex["label"], justification=ex["justification"])
    neg3_records.append({
        "id":           ex["id"],
        "neg_type":     "neg3_hedge",
        "passage":      ex["passage"],
        "claim":        ex["claim"],
        "ground_truth": ex["label"],
        "rejected_raw": rejected,
    })

print(f"NEG3 constructed: {len(neg3_records)}")
print("sample:", neg3_records[0]["rejected_raw"][:150])

NEG3 constructed: 250
sample: This seems to be NOT MENTIONED: The passage discusses a decision related to the Russian Orthodox Church and the Raskol, but it does not mention Faith 


---
## NEG4 — Circular Citing (GPT-4.1-mini)

Correct label, but justification argues for the verdict using only information from the **claim itself** — never citing or quoting the passage. Sounds confident but is circular.

Targets the failure mode where the model produces a plausible-sounding response that isn't actually grounded in the evidence.

In [24]:
NEG4_SYSTEM = """You are generating REJECTED responses for preference optimization training.

Write a single fact-checking response in the format: LABEL: one-sentence justification

Rules:
- Use the correct label provided.
- Write a justification that argues for the verdict but only references information from the claim itself, never citing or quoting the passage.
- The justification should sound confident but be circular.

Output only the single response line."""

def get_circular_negative(passage, claim, label):
    user_msg = f"Passage: {passage}\n\nClaim: {claim}\n\nCorrect label: {label}"
    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=TEACHER_DEPLOYMENT,
                messages=[
                    {"role": "system", "content": NEG4_SYSTEM},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0.7,
                max_tokens=120,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return None

In [25]:
# spot-check
for true_label in ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]:
    ex = random.choice([e for e in ef_data if e["label"] == true_label])
    result = get_circular_negative(ex["passage"], ex["claim"], ex["label"])
    print(f"label   : {ex['label']}")
    print(f"claim   : {ex['claim']}")
    print(f"chosen  : {ex['label']}: {ex['justification']}")
    print(f"rejected: {result}")
    print()

label   : SUPPORTED
claim   : Paranormal pertains to ESP abilities.
chosen  : SUPPORTED: The passage states that paranormal beliefs include psychic abilities or extrasensory perception (ESP), which supports the claim that paranormal pertains to ESP abilities.
rejected: SUPPORTED: The claim that paranormal pertains to ESP abilities is supported because the passage explicitly includes extrasensory perception (ESP) as a notable paranormal belief.

label   : CONTRADICTED
claim   : Steve Buscemi has appeared in only one film by the Coen Brothers.
chosen  : CONTRADICTED: The passage states that Steve Buscemi is known for his appearances in many films by the Coen brothers, listing Miller's Crossing, Barton Fink, The Hudsucker Proxy, Fargo, and The Big Lebowski, which contradicts the claim that he has appeared in only one film by them.
rejected: CONTRADICTED: The claim states Steve Buscemi appeared in only one Coen brothers film, but the passage lists multiple films he appeared in by them, dir

In [26]:
NEG4_OUT = "data/generated/dpo_v2_neg4.jsonl"

neg4_pool = random.sample(ef_data, 500)
done_ids = set()
if os.path.exists(NEG4_OUT):
    with open(NEG4_OUT) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
    print(f"resuming, {len(done_ids)} already done")

skipped = 0
with open(NEG4_OUT, "a") as out_f:
    for i, ex in enumerate(neg4_pool):
        if ex["id"] in done_ids:
            continue
        rejected = get_circular_negative(ex["passage"], ex["claim"], ex["label"])
        if rejected is None:
            skipped += 1
            continue
        record = {
            "id":           ex["id"],
            "neg_type":     "neg4_circular",
            "passage":      ex["passage"],
            "claim":        ex["claim"],
            "ground_truth": ex["label"],
            "rejected_raw": rejected,
        }
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()
        if (i + 1) % 100 == 0:
            print(f"[{i+1}/{len(neg4_pool)}] skipped={skipped}")

print(f"NEG4 done. skipped {skipped} → {NEG4_OUT}")

[100/500] skipped=0
[200/500] skipped=0
[300/500] skipped=0
[400/500] skipped=0
[500/500] skipped=0
NEG4 done. skipped 0 → data/generated/dpo_v2_neg4.jsonl


---
## NEG5 — Degenerate Output (Constructed)

Programmatically constructed broken responses that mimic real model failures observed in dpo_c1/c3:
- Repetition loops ("no no no no", "con con con")
- Truncated mid-sentence output
- Missing or lowercase label
- Garbled label prefix ("CONCON:", "V.V.:")

In [27]:
DEGENERATE_TEMPLATES = [
    # repetition loops — "no no no no"
    lambda lbl, just: "no " * random.randint(15, 30) + just[:40],
    # con/CONCON prefix repetition before colon (observed in dpo_c1)
    lambda lbl, just: lbl[:3].lower() + (lbl[:3].upper() + lbl[:3].lower()) * random.randint(5, 10) + ": " + just[:60],
    # truncated mid-sentence
    lambda lbl, just: f"{lbl}: " + just[:random.randint(10, 25)],
    # lowercase label
    lambda lbl, just: f"{lbl.lower()}: {just}",
    # title-case label only (e.g. "Supported: ...")
    lambda lbl, just: f"{lbl.capitalize()}: {just}",
    # label-less: starts mid-sentence with "supported by the passage..." (observed in dpo_c1)
    lambda lbl, just: f"{lbl.lower()} by the passage that {just}",
    # label-less: starts directly with justification, no label at all
    lambda lbl, just: just,
    # garbled prefix (e.g. "CONCON:", "SUPSUPSUP:")
    lambda lbl, just: (lbl[:3] * random.randint(2, 4)) + ": " + just,
]

neg5_pool = random.sample(ef_data, 250)
neg5_records = []
for ex in neg5_pool:
    template = random.choice(DEGENERATE_TEMPLATES)
    rejected = template(ex["label"], ex["justification"])
    neg5_records.append({
        "id":           ex["id"],
        "neg_type":     "neg5_degenerate",
        "passage":      ex["passage"],
        "claim":        ex["claim"],
        "ground_truth": ex["label"],
        "rejected_raw": rejected,
    })

print(f"NEG5 constructed: {len(neg5_records)}")
for r in random.sample(neg5_records, 5):
    print(" ", repr(r["rejected_raw"][:100]))

NEG5 constructed: 250
  "supported: The passage states that the song was featured on the group's eighth studio album The Game"
  'notNOTnotNOTnotNOTnotNOTnotNOTnotNOTnot: The passage states that "He commanded the 5th Infantry Divi'
  'no no no no no no no no no no no no no no no no no no no no no no no no no no no no no The passage s'
  'The passage states that Karan Johar\'s film "My Name Is Khan" earned him a Filmfare Award for Best Di'
  'conCONconCONconCONconCONconCONconCONconCONcon: The passage states that "الإسلام ... is an Abrahamic '


---
## NEG6 — Reasoning-Label Mismatch (Constructed)

Take a correct justification and attach the **wrong label** — the reasoning leads clearly to the right verdict but the label contradicts it.

This is the most dangerous failure mode observed in dpo_c3: the model writes a justification that correctly identifies a contradiction but then outputs SUPPORTED, or correctly grounds support but outputs CONTRADICTED. DPO directly penalizes this by contrasting identical justifications with right vs wrong labels.

In [28]:
OTHER_LABELS = {
    "SUPPORTED":     ["CONTRADICTED", "NOT MENTIONED"],
    "CONTRADICTED":  ["SUPPORTED",    "NOT MENTIONED"],
    "NOT MENTIONED": ["SUPPORTED",    "CONTRADICTED"],
}

neg6_pool = random.sample(ef_data, 500)
neg6_records = []
for ex in neg6_pool:
    wrong_label = random.choice(OTHER_LABELS[ex["label"]])
    # same justification, wrong label — the justification clearly points to the correct verdict
    rejected = f"{wrong_label}: {ex['justification']}"
    neg6_records.append({
        "id":           ex["id"],
        "neg_type":     "neg6_mismatch",
        "passage":      ex["passage"],
        "claim":        ex["claim"],
        "ground_truth": ex["label"],
        "rejected_raw": rejected,
    })

print(f"NEG6 constructed: {len(neg6_records)}")
print("sample:")
s = neg6_records[0]
ex = ef_by_id[s["id"]]
print(f"  chosen  : {ex['label']}: {ex['justification']}")
print(f"  rejected: {s['rejected_raw']}")

NEG6 constructed: 500
sample:
  chosen  : SUPPORTED: The passage states that Christoph Waltz "received the Best Actor Award at the Cannes Film Festival," which directly supports the claim that he received an award in the Best Actor category.
  rejected: CONTRADICTED: The passage states that Christoph Waltz "received the Best Actor Award at the Cannes Film Festival," which directly supports the claim that he received an award in the Best Actor category.


---
## Assemble Final DPO v2 Pairs

Combine all 6 negative types. Chosen side is always the EF-style justification from `sft_data_ef.jsonl`.
NEG1/NEG2 chosen side falls back to `sft_data_3k.jsonl` since they were generated from that pool.

In [29]:
neg4_records = load_jsonl(NEG4_OUT)[:500]
print(f"NEG4 loaded: {len(neg4_records)}")

# also need sft_data_3k for neg1/neg2 chosen fallback
sft_3k_by_id = {ex["id"]: ex for ex in load_jsonl("data/generated/sft_data_3k.jsonl")}

NEG4 loaded: 500


In [30]:
SYSTEM_PROMPT = """You are a fact-checking assistant. Given a passage and a claim, respond with the verdict followed by a one-sentence justification quoting or closely paraphrasing the passage.
Format: LABEL: justification sentence
Label must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED"""

DPO_V2_OUT = "data/generated/dpo_data_v2_3k.jsonl"

def get_rejected_text(neg_record):
    if "rejected_raw" in neg_record:
        return neg_record["rejected_raw"]
    # NEG1/NEG2 format
    return f"{neg_record['rejected_label']}: {neg_record['rejected_justification']}"

def make_pair(neg_record, chosen_ex, neg_type):
    chosen_response = f"{chosen_ex['label']}: {chosen_ex['justification']}"
    return {
        "id":       neg_record["id"],
        "neg_type": neg_type,
        "prompt":   f"Passage: {neg_record['passage']}\n\nClaim: {neg_record['claim']}",
        "chosen":   chosen_response,
        "rejected": get_rejected_text(neg_record),
    }

neg1_sample = random.sample(neg1_raw, min(750, len(neg1_raw)))
neg2_sample = random.sample(neg2_raw, min(750, len(neg2_raw)))

all_negs = [
    (neg1_sample,  sft_3k_by_id, "neg1_wrong_label"),
    (neg2_sample,  sft_3k_by_id, "neg2_bad_reasoning"),
    (neg3_records, ef_by_id,     "neg3_hedge"),
    (neg4_records, ef_by_id,     "neg4_circular"),
    (neg5_records, ef_by_id,     "neg5_degenerate"),
    (neg6_records, ef_by_id,     "neg6_mismatch"),
]

dpo_pairs = []
skipped = 0
for neg_list, chosen_pool, neg_type in all_negs:
    for neg in neg_list:
        chosen_ex = chosen_pool.get(neg["id"])
        if chosen_ex is None:
            skipped += 1
            continue
        dpo_pairs.append(make_pair(neg, chosen_ex, neg_type))

random.shuffle(dpo_pairs)

with open(DPO_V2_OUT, "w") as f:
    for pair in dpo_pairs:
        f.write(json.dumps(pair) + "\n")

print(f"Total DPO v2 pairs: {len(dpo_pairs)}  (skipped {skipped}) → {DPO_V2_OUT}")
print()
print("Breakdown by neg type:")
print(Counter(p["neg_type"] for p in dpo_pairs))

Total DPO v2 pairs: 3000  (skipped 0) → data/generated/dpo_data_v2_3k.jsonl

Breakdown by neg type:
Counter({'neg1_wrong_label': 750, 'neg2_bad_reasoning': 750, 'neg4_circular': 500, 'neg6_mismatch': 500, 'neg5_degenerate': 250, 'neg3_hedge': 250})


In [31]:
# sanity check: one sample per neg type
by_type = defaultdict(list)
for p in dpo_pairs:
    by_type[p["neg_type"]].append(p)

for neg_type, pairs in sorted(by_type.items()):
    s = random.choice(pairs)
    print(f"=== {neg_type} ===")
    print(f"  chosen  : {s['chosen']}")
    print(f"  rejected: {s['rejected']}")
    print()

=== neg1_wrong_label ===
  chosen  : SUPPORTED: The passage states that FC Barcelona was "Founded in 1899 by a group of Swiss, English and Catalan footballers," which supports the claim that it was formed by a group of local (Catalan), Swiss, and English players.
  rejected: CONTRADICTED: The passage states that the 2016 Summer Olympics were "held in Rio de Janeiro, Brazil," directly contradicting the claim.

=== neg2_bad_reasoning ===
  chosen  : NOT MENTIONED: The passage only states that "It was described by Matsumura in 1930" and does not provide any information about Nepal's size or ranking among countries.
  rejected: NOT MENTIONED: The claim is supported because the passage mentions a specific date and author, implying detailed knowledge about geographical classifications.

=== neg3_hedge ===
  chosen  : SUPPORTED: The passage states, "In 70, he besieged and captured Jerusalem, and destroyed the city and the Second Temple," directly confirming that Titus destroyed the Second Tem